# Finals: Digit Formations — Full Kaggle Pipeline

Generates assignment + setpoint datasets, pretrains LocalNegotiator, trains Bertsekas + SetpointGATv2, zips artifacts.

**Kaggle**: GPU **T4 ×2 accelerator** OK — the notebook exposes **only one GPU** (`CUDA_VISIBLE_DEVICES=0`) so PyTorch/PyG **never enables `nn.DataParallel`** (avoids duplicate-import / IndexError). Internet ON.

**Outputs**: `/kaggle/working/merged_artifacts/`.

• **Warm-up**: Leave `RUN_GPU_SMOKE = True`; first run completes §4.5 (minutes) plus `TINY_TEST = True` end-to-end.
• **Overnight**: after warm-up succeeds, set `RUN_GPU_SMOKE = False` and `TINY_TEST = False`, then **Save Version → Save & Run All**.

Full walkthrough: [`../KAGGLE_RUN.md`](../KAGGLE_RUN.md)

In [ ]:
# Pin one physical GPU BEFORE any Torch import — avoids nn.DataParallel + PyG IndexError on Kaggle (2×T4 kernels).
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES=0 (single GPU for whole session)")


In [ ]:
import importlib, subprocess, sys


def _ver(name):
    try:
        return importlib.import_module(name).__version__
    except Exception:
        return None


import numpy

np_major = int(numpy.__version__.split(".")[0])
scipy_spec = "scipy>=1.13,<1.15" if np_major >= 2 else "scipy>=1.11,<1.14"


def _scipy_works():
    try:
        for k in list(sys.modules):
            if k == "scipy" or k.startswith("scipy."):
                del sys.modules[k]
        from scipy.optimize import linear_sum_assignment  # noqa: F401

        return True
    except Exception as e:
        print("scipy broken:", e)
        return False


if not _scipy_works():
    print(f"Installing {scipy_spec} for numpy {numpy.__version__}")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--force-reinstall",
            "--no-deps",
            scipy_spec,
        ]
    )
    for k in list(sys.modules):
        if k == "scipy" or k.startswith("scipy."):
            del sys.modules[k]
    from scipy.optimize import linear_sum_assignment  # noqa: F401

print(f"numpy {numpy.__version__} | scipy {_ver('scipy')} OK")

## 1. Paths & imports

In [ ]:
import os, sys
from pathlib import Path

# Detect repo root (notebook in merged_work/models_creation OR Kaggle dataset input)
NOTEBOOK_DIR = Path.cwd()
REPO = None

for hint in (NOTEBOOK_DIR, Path("/kaggle/input")):
    if not hint.exists():
        continue
    for p in hint.rglob("models_creation"):
        if (p / "finals_pipeline.py").exists():
            REPO = p.parent.parent
            break
    if REPO is not None:
        break

assert REPO and (REPO / "merged_work" / "models_creation").exists(), (
    f"merged_work not found under {REPO}. "
    "Ensure your gnn_drone_project zip is attached as a Kaggle Dataset input."
)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

WORK = Path("/kaggle/working/merged_artifacts") if Path("/kaggle/working").exists() else REPO / "merged_artifacts"
WORK.mkdir(parents=True, exist_ok=True)
print("REPO:", REPO)
print("WORK:", WORK)

## 2. Install dependencies (PyG, PyFlyt, libgl1)

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

# Headless GL stubs for PyBullet (Kaggle)
if Path("/kaggle").is_dir():
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "libgl1-mesa-glx"],
        check=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )


def _have(mod: str) -> bool:
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False


for mod, pip_name in [
    ("scipy", "scipy"),
    ("safetensors", "safetensors"),
    ("tqdm", "tqdm"),
    ("pybullet", "pybullet"),
    ("torch_geometric", "torch-geometric"),
]:
    if not _have(mod):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

# PyFlyt imports as PyFlyt.core, not pyflyt
if not _have("PyFlyt.core"):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pyflyt"]
    )

HAS_PYFLYT = False
try:
    from PyFlyt.core import Aviary  # noqa: F401

    HAS_PYFLYT = True
    print("PyFlyt OK — setpoint physics rollouts enabled")
except Exception as exc:
    print(
        "PyFlyt unavailable:",
        exc,
        "→ setpoint dataset + setpoint GNN will be SKIPPED. Assignment models will still train.",
    )

## 3. Patch Bertsekas import + preflight

In [ ]:
import importlib

from merged_work.models_creation.package_setup import setup_project_paths

setup_project_paths()

ln = importlib.import_module("local_negotiator_v2")
ba = importlib.import_module("bertsekas_auction_v2")
if not hasattr(ba, "train_strict_decentralized_model"):
    ba.train_strict_decentralized_model = ln.train_strict_decentralized_model
print("Patched bertsekas_auction_v2.train_strict_decentralized_model")

from merged_work.models_creation.finals_pipeline import preflight, run_pipeline, smoke_test
preflight()

## 4. Configuration (TINY_TEST toggle)

In [ ]:
# Warm-up defaults: verifies stack + short setpoint epochs. Overnight: RUN_GPU_SMOKE=False, TINY_TEST=False.
RUN_GPU_SMOKE = True

TINY_TEST = True

if TINY_TEST:
    CFG = dict(
        assignment_episodes=200,
        setpoint_episodes=4,
        setpoint_workers=1,
        localneg_epochs=2,
        bertsekas_epochs=2,
        setpoint_cfg={"epochs": 2, "patience": 2},
    )
    print(">>> MODE: TINY_TEST (dry-run incl. ~2-epoch setpoint train; ~10-25 min)")
else:
    CFG = dict(
        assignment_episodes=3000,
        setpoint_episodes=500,
        setpoint_max_steps=400,
        setpoint_workers=1,
        localneg_epochs=60,
        bertsekas_epochs=80,
    )
    print(">>> MODE: FULL RUN (single GPU; 500 setpoint eps × 400 max steps; v2 imitation)")

CFG["has_pyflyt"] = HAS_PYFLYT
CFG["skip_setpoint_data"] = not HAS_PYFLYT
CFG

## 4.5 Setpoint GPU smoke (optional; catches PyG / setpoint training errors in minutes)

In [ ]:
import shutil

import torch


def _gpu_smoke():
    assert torch.cuda.is_available(), "GPU smoke needs CUDA (use Kaggle GPU kernel)"
    assert torch.cuda.device_count() == 1, (
        f"Expected 1 visible GPU after CUDA pin, got {torch.cuda.device_count()}"
    )
    print("GPU smoke device:", torch.cuda.get_device_name(0))

    from merged_work.models_creation.dataset_runner import generate_setpoint_dataset
    from merged_work.models_creation.setpoint_training import TRAIN_CFG, train_setpoint_v3

    smoke_dir = WORK / "smoke_setpoint_gpu"
    if smoke_dir.exists():
        shutil.rmtree(smoke_dir)
    smoke_dir.mkdir(parents=True, exist_ok=True)

    generate_setpoint_dataset(
        smoke_dir,
        num_episodes=3,
        num_workers=1,
        max_steps=80,
        save_interval=5,
    )

    smoke_cfg = dict(TRAIN_CFG)
    smoke_cfg["epochs"] = 2
    smoke_cfg["patience"] = 2

    ck_smoke = smoke_dir / "checkpoints"

    train_setpoint_v3(
        smoke_dir / "setpoint_digits_train.pt",
        smoke_dir / "setpoint_digits_val.pt",
        smoke_dir / "setpoint_digits_test.pt",
        ck_smoke,
        torch.device("cuda"),
        cfg=smoke_cfg,
        use_data_parallel=False,
    )
    print(">>> GPU SMOKE PASSED")


if HAS_PYFLYT and RUN_GPU_SMOKE:
    _gpu_smoke()
elif not HAS_PYFLYT:
    print("[skip] GPU smoke — PyFlyt unavailable")
else:
    print("[skip] GPU smoke — RUN_GPU_SMOKE is False")


## 5. Run pipeline (resumable)

In [ ]:
# Patch BOTH module bindings — `run_pipeline` holds a stale reference to train_setpoint_v3 otherwise.
import merged_work.models_creation.finals_pipeline as _fp
import merged_work.models_creation.setpoint_training as _st

_orig_train_sp = _fp.train_setpoint_v3


def _train_setpoint_single_gpu(*args, **kwargs):
    kwargs["use_data_parallel"] = False
    return _orig_train_sp(*args, **kwargs)


_fp.train_setpoint_v3 = _train_setpoint_single_gpu
_st.train_setpoint_v3 = _train_setpoint_single_gpu
print("Setpoint training: enforced single-GPU (finals_pipeline + setpoint_training bindings)")

# Every setpoint rollout uses path + slot obstacles (does not change assignment dataset).
import merged_work.models_creation.setpoint_rollout as _sr
from merged_work.models_creation.dataset_pipeline import make_episode_config

_orig_rollout = _sr.rollout_from_seed


def _rollout_both(ep_idx, split, seed, num_drones, **kwargs):
    cfg = make_episode_config(seed=seed, num_drones=num_drones, scenario="both")
    return _sr.simulate_setpoint_episode(ep_idx, split, cfg, **kwargs)


_sr.rollout_from_seed = _rollout_both
print("Setpoint rollouts pinned to scenario=both")

zip_path = run_pipeline(WORK, **CFG)
print("Artifact:", zip_path)

## 6. Smoke test + digit graph check (optional)

In [ ]:
smoke_test(WORK)

from merged_work.models_creation.dataset_pipeline import (
    assign_drones_to_slots,
    build_assignment_graph,
    build_naive_slots,
    make_episode_config,
    sample_initial_state,
)

cfg = make_episode_config(seed=123, num_drones=12, digit=7, scenario="both")
start_pos, _ = sample_initial_state(cfg)
slots = build_naive_slots(cfg, start_pos)
assignment = assign_drones_to_slots(start_pos, slots)
print(build_assignment_graph(cfg, start_pos, slots, assignment))